In [1]:
import pathlib, sys
import matplotlib
matplotlib.use('Agg')
_root = pathlib.Path.cwd()
if not (_root / 'src').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# 01 · The Ritual Pipeline

Every NLP tutorial opens the same way: lowercase, strip punctuation, remove stopwords, stem or lemmatize, drop numbers. This notebook applies that ritual **step by step** so you can see exactly what each step does to the text, and measures the vocabulary reduction each one buys — the historical justification for the whole exercise.

The question the rest of the arena answers: this pipeline was designed to help *bag-of-words* models. Does it still help the models we actually use?

In [2]:
import pandas as pd
from src.cleaning.steps import STEPS
from src.cleaning.pipelines import PIPELINE_NAMES, clean
from src.cleaning.analysis import step_table, pipeline_table
from src.datasets.corpora import load_sentiment
from src.visualisation import plot_vocab_reduction, save_figure, save_json
pd.set_option('display.max_colwidth', 90)
print('pipelines:', PIPELINE_NAMES)

pipelines: ('none', 'minimal', 'standard', 'aggressive')


## Each step, applied to one sentence

Watch the sentence degrade. Note the casing collapse, the deleted `NOT`, and the stemmer turning real words into non-words.

In [3]:
sentence = "The Movie was NOT great, I would never watch these 2 films again!"
steps_demo = pd.DataFrame([{'step': name, 'result': fn(sentence)} for name, fn in STEPS.items()])
save_json({'sentence': sentence, 'steps': steps_demo.to_dict(orient='records')}, 'nb01_steps_on_sentence.json')
steps_demo

,step,result
0,normalize_whitespace,"The Movie was NOT great, I would never watch these 2 films again!"
1,lowercase,"the movie was not great, i would never watch these 2 films again!"
2,remove_stopwords,"Movie great, would watch 2 films"
3,strip_punctuation,The Movie was NOT great I would never watch these 2 films again
4,remove_numbers,"The Movie was NOT great, I would never watch these films again!"
5,stem,"the movi wa not great, i would never watch these 2 film again!"
6,lemmatize,"The Movie wa NOT great, I would never watch these 2 film again!"


## The four named pipelines

`none` (raw), `minimal` (whitespace only), `standard` (lowercase + stopwords + lemmatize), `aggressive` (everything + stemming + number removal).

In [4]:
examples = [
    'The Movie was NOT great, I would never watch it again!',
    'Apple shares rose as the US economy grew 3% in March.',
    'Running shoes: the studies show they help runners run faster.',
]
rows = []
for ex in examples:
    row = {'original': ex}
    row.update({p: clean(ex, p) for p in PIPELINE_NAMES})
    rows.append(row)
pipelines_demo = pd.DataFrame(rows)
save_json(pipelines_demo, 'nb01_pipelines_on_sentences.json')
pipelines_demo

,original,none,minimal,standard,aggressive
0,"The Movie was NOT great, I would never watch it again!","The Movie was NOT great, I would never watch it again!","The Movie was NOT great, I would never watch it again!","movie great, would watch",movi great would watch
1,Apple shares rose as the US economy grew 3% in March.,Apple shares rose as the US economy grew 3% in March.,Apple shares rose as the US economy grew 3% in March.,apple share rose economy grew 3% march.,appl share rose economi grew march
2,Running shoes: the studies show they help runners run faster.,Running shoes: the studies show they help runners run faster.,Running shoes: the studies show they help runners run faster.,running shoe: study show help runner run faster.,run shoe studi show help runner run faster


## What each step buys: vocabulary reduction

The historical payoff of cleaning is a smaller, denser vocabulary for sparse models. Measured on the sentiment corpus.

In [5]:
texts, _ = load_sentiment()
step_reports = step_table(texts)
step_df = pd.DataFrame([
    {'step': r.name, 'vocab_before': r.vocab_before, 'vocab_after': r.vocab_after,
     'vocab_reduction_pct': round(r.vocab_reduction_pct, 1), 'negations_removed': r.negations_removed}
    for r in step_reports])
save_json(step_df, 'nb01_vocab_reduction_by_step.json')
fig = plot_vocab_reduction(step_reports)
save_figure(fig, 'nb01_vocab_reduction_by_step.png')
step_df

,step,vocab_before,vocab_after,vocab_reduction_pct,negations_removed
0,normalize_whitespace,81,81,0.0,0
1,lowercase,81,81,0.0,0
2,remove_stopwords,81,65,19.8,33
3,strip_punctuation,81,81,0.0,0
4,remove_numbers,81,81,0.0,0
5,stem,81,79,2.5,0
6,lemmatize,81,81,0.0,0


In [6]:
pipe_reports = pipeline_table(texts)
pipe_df = pd.DataFrame([
    {'pipeline': r.name, 'vocab_before': r.vocab_before, 'vocab_after': r.vocab_after,
     'vocab_reduction_pct': round(r.vocab_reduction_pct, 1), 'negations_removed': r.negations_removed}
    for r in pipe_reports])
save_json(pipe_df, 'nb01_vocab_reduction_by_pipeline.json')
fig = plot_vocab_reduction(pipe_reports)
save_figure(fig, 'nb01_vocab_reduction_by_pipeline.png')
pipe_df

,pipeline,vocab_before,vocab_after,vocab_reduction_pct,negations_removed
0,none,81,81,0.0,0
1,minimal,81,81,0.0,0
2,standard,81,65,19.8,33
3,aggressive,81,63,22.2,33


## Takeaway

Each step shrinks the vocabulary — exactly what a sparse bag-of-words model wants. But notice the `negations_removed` column: stopword removal (and every pipeline that includes it) deletes `not` / `no` / `never`. That is free vocabulary reduction for an old model and destroyed signal for a new one. The next notebook shows the damage on real sentences.